# Create RWJF Awards

Creates Robert Wood Johnson Foundation grants from rwjf.org's grants
explorer (~31,717 grants totaling $14.4B across 1972-2026), scraped
via Playwright + headless Chromium (the page is JS-rendered with no
public API).

**Prerequisites:**
- Run `scripts/local/rwj_to_s3.py` (Playwright) to scrape pages and
  upload the parquet first. ~2,115 pages × ~15 grants × ~15s/page =
  approx 9 hours full-corpus run; checkpoints every 50 pages.

**Data source:** rwjf.org/en/grants/awarded-grants.html?s=N
**S3 location:** `s3a://openalex-ingest/awards/rwj/rwj_grants.parquet`

**RWJF funder (OpenAlex):**
- funder_id: 4320306139
- ROR: https://ror.org/02ymmdj85
- DOI: 10.13039/100000867
- display_name: "Robert Wood Johnson Foundation"

**Schema notes:**
- Listing exposes Grant Title, Location, Year Awarded, Amount Awarded,
  and (sometimes) Program Area / Areas. No PI name; the grant title
  itself is descriptive.
- ~80% of grants have a Program Area; the rest don't display one
  publicly.
- Some grants have NULL Amount Awarded (small grants, transfer
  grants, or RWJF President's Discretionary Fund draws).
- Currency hardcoded USD.


## Step 1: Create staging table from S3

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rwj_raw
USING delta
AS
SELECT *, current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/rwj/rwj_grants.parquet`;

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.rwj_raw;

## Step 1.5: Inspect raw + verify amount/currency

In [ ]:
%sql
DESCRIBE openalex.awards.rwj_raw;

In [ ]:
%sql
SELECT * FROM openalex.awards.rwj_raw LIMIT 5;

In [ ]:
%sql
SELECT
  COUNT(*) AS rows,
  COUNT(amount_usd) AS has_amount,
  ROUND(COUNT(amount_usd) * 100.0 / COUNT(*), 1) AS pct_amount,
  ROUND(MIN(amount_usd), 0) AS min_amt,
  ROUND(MAX(amount_usd), 0) AS max_amt,
  ROUND(AVG(amount_usd), 0) AS avg_amt,
  COUNT(DISTINCT program_area) AS distinct_program_areas
FROM openalex.awards.rwj_raw;

## Step 2: Transform to award schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.rwj_awards
USING delta
AS
WITH rwj_funder AS (
    SELECT funder_id, display_name, ror_id, doi
    FROM openalex.common.funder
    WHERE funder_id = 4320306139
),
src AS (
    SELECT
        *,
        abs(xxhash64(CONCAT(
            COALESCE(grant_title, ''), ':',
            CAST(COALESCE(year_awarded, 0) AS STRING), ':',
            COALESCE(CAST(amount_usd AS STRING), ''), ':',
            COALESCE(location, ''), ':',
            CAST(page AS STRING), ':',
            CAST(monotonically_increasing_id() AS STRING)
        ))) % 9000000000 AS surrogate_id
    FROM openalex.awards.rwj_raw
    WHERE grant_title IS NOT NULL AND TRIM(grant_title) != ''
)
SELECT
    abs(xxhash64(CONCAT(f.funder_id, ':rwj:', CAST(s.surrogate_id AS STRING)))) % 9000000000 AS id,
    s.grant_title AS display_name,
    CAST(NULL AS STRING) AS description,
    f.funder_id,
    CAST(s.surrogate_id AS STRING) AS funder_award_id,
    s.amount_usd AS amount,
    'USD' AS currency,
    struct(
        CONCAT('https://openalex.org/F', f.funder_id) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    'grant' AS funding_type,
    -- Program Area / Areas (multi-line in source; collapse to single string)
    CASE WHEN s.program_area IS NOT NULL AND TRIM(s.program_area) != ''
         THEN regexp_replace(s.program_area, '[\\n\\r]+', ' / ')
         ELSE NULL END AS funder_scheme,
    'rwjf_grants_explorer' AS provenance,
    CASE WHEN s.year_awarded IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(CAST(s.year_awarded AS STRING), '-01-01'), 'yyyy-MM-dd') END AS start_date,
    CASE WHEN s.year_awarded IS NOT NULL
         THEN TRY_TO_DATE(CONCAT(CAST(s.year_awarded AS STRING), '-12-31'), 'yyyy-MM-dd') END AS end_date,
    s.year_awarded AS start_year,
    s.year_awarded AS end_year,
    struct(
        CAST(NULL AS STRING) AS given_name,
        CAST(NULL AS STRING) AS family_name,
        CAST(NULL AS STRING) AS orcid,
        CAST(NULL AS DATE) AS role_start,
        struct(
            CAST(NULL AS STRING) AS name,  -- listing has location only, not org
            -- Last token of "City, ST" is the state/country code
            element_at(split(s.location, ', '), -1) AS country,
            CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
        ) AS affiliation
    ) AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    CONCAT('https://www.rwjf.org/en/grants/awarded-grants.html?s=', CAST(s.page AS STRING)) AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    concat('https://api.openalex.org/works?filter=awards.id:G',
           abs(xxhash64(CONCAT(f.funder_id, ':rwj:', CAST(s.surrogate_id AS STRING)))) % 9000000000) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM src s
CROSS JOIN rwj_funder f;

## Step 3: Insert into openalex_awards_raw at priority 46

In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'rwjf_grants_explorer' AND priority = 46;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    46 AS priority
FROM openalex.awards.rwj_awards;

## Verification

In [ ]:
%sql
SELECT COUNT(*) FROM openalex.awards.rwj_awards;

In [ ]:
%sql
-- Step 6.7 fail-fast amount/currency check
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(COUNT(amount) * 100.0 / COUNT(*), 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    ROUND(MIN(amount), 0) AS min_amt,
    ROUND(MAX(amount), 0) AS max_amt,
    ROUND(AVG(amount), 0) AS avg_amt,
    ROUND(SUM(amount), 0) AS total_funding_usd
FROM openalex.awards.rwj_awards;

In [ ]:
%sql
SELECT id, SUBSTRING(display_name, 1, 80) AS title, amount, start_year, funder_scheme
FROM openalex.awards.rwj_awards
ORDER BY amount DESC LIMIT 10;

In [ ]:
%sql
SELECT funder_scheme AS program_area, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.rwj_awards
WHERE funder_scheme IS NOT NULL
GROUP BY funder_scheme
ORDER BY cnt DESC LIMIT 15;

In [ ]:
%sql
SELECT start_year, COUNT(*) AS cnt, ROUND(SUM(amount), 0) AS total_usd
FROM openalex.awards.rwj_awards
WHERE start_year IS NOT NULL
GROUP BY start_year ORDER BY start_year DESC LIMIT 20;